In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, lora_alpha=16, lora_dropout=0.05):
        super().__init__()
        self.pretrained = nn.Linear(in_features, out_features)  # 实际应用中，这里应该是将预训练好的Linear层的weights和bias拿过来放进self.pretrained中

        # 冻结预训练参数
        self.pretrained.weight.requires_grad = False
        if self.pretrained.bias is not None:
            self.pretrained.bias.requires_grad = False

        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r

        if r > 0:
            self.lora_dropout = nn.Dropout(p=lora_dropout)

            self.lora_A = nn.Linear(in_features, r, bias=False)
            self.lora_B = nn.Linear(r, out_features, bias=False)

            self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        # B初始化为全零，保证初始状态下LoRA旁路输出为0
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        # y = x * W_new = x * [W_old + (lora_B * lora_A)] = x * W_old + x * (lora_B * lora_A)
        # 因此LoRA的前向传播可以
        result = self.pretrained(x)
        
        if self.r > 0:
            lora_result = self.lora_B(self.lora_A(self.lora_dropout(x)))
            result = result + lora_result * self.scaling
        
        return result


In [ ]:
import torch.nn.functional as F
def inject_pretrained_weights(original_linear: nn.Linear, r=8):
    """
    这个函数接收一个原版的、带有预训练权重的 Linear 层，
    返回一个完美继承了它权重的 LoRALinear 层。
    """
    in_features = original_linear.in_features
    out_features = original_linear.out_features
    
    # 1. 实例化我们的 LoRA 外挂层 (此时它内部的 pretrained 是随机初始化的)
    lora_layer = LoRALinear(in_features, out_features, r=r)
    
    # 2. 数据搬运
    # 使用 .clone() 安全地拷贝张量数据，防止内存指针混乱
    lora_layer.pretrained.weight.data = original_linear.weight.data.clone()
    
    # 如果原模型有偏置项 (Bias)，也一并搬过去
    if original_linear.bias is not None:
        lora_layer.pretrained.bias.data = original_linear.bias.data.clone()
        
    # 3. 冻结预训练权重，不参与梯度更新
    lora_layer.pretrained.weight.requires_grad = False
    if lora_layer.pretrained.bias is not None:
        lora_layer.pretrained.bias.requires_grad = False
        
    return lora_layer

# 带LoRA的MHA
class LoRAMultiHeadAttention(nn.Module):
    def __init__(self, 
                 d_model=512, 
                 num_heads=8, 
                 r=8, 
                 lora_alpha=16):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 先随机生成pretrained，
        self.W_Q = LoRALinear(d_model, d_model, r=r, lora_alpha=lora_alpha)
        self.W_K = LoRALinear(d_model, d_model, r=r, lora_alpha=lora_alpha)
        self.W_V = LoRALinear(d_model, d_model, r=r, lora_alpha=lora_alpha)
        self.W_O = LoRALinear(d_model, d_model, r=r, lora_alpha=lora_alpha)

    def forward(self, q, k, v, mask=None):
        batch_size = q.shape[0]
        Q = self.W_Q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        K_transposed = K.transpose(-1, -2)

        score = torch.matmul(Q, K_transposed) / math.sqrt(self.d_k)

        if mask is not None:
            score = score.masked_fill(mask == 0, -1e9)

        attn_weights = F.softmax(score, dim=-1)
        context = torch.matmul(attn_weights, V)

        context = context.transpose(1, 2)
        context = context.contiguous().view(batch_size, -1, self.d_model)

        return self.W_O(context)





### 1. 关于 `r <= 0`：如果不激活旁路，是不是就跟普通 Linear 一样？
**绝对正确！**
如果 `r <= 0`，代码里的 `if r > 0:` 分支就不会执行。数据直接流过 `self.pretrained`。
这在工程上非常有用：有时候我们在同一个代码库里，既想做全量微调，又想做 LoRA 微调，通过把 `r` 设为 0，这套代码就能瞬间回退退化成一个普通的、甚至可以被全量更新的 Linear 层。

---

### 2. 关于初始化：为什么要保证第一步表现完全一样？为什么是 B=0 而不是 A=0？

**问题 A：为什么要保证微调第一步和没加之前完全一样？**
大模型（比如 LLaMA）是耗费几千万美金、看了上万亿个单词才炼出来的，它内部的参数极其精妙且脆弱。
如果在微调的第 1 步，你加的 LoRA 旁路输出了毫无意义的随机噪点（Noise），这些噪点和原模型的输出相加后，会瞬间破坏掉大模型原本极其完美的语义表示。这会导致 Loss 在第一步直接爆炸，模型瞬间变“傻”，你要花极其漫长的时间才能让它重新收敛。
所以，**微调的第一准则是：Do No Harm（不作恶）**。初始状态的 $\Delta W$ 必须是 0。

**问题 B：为什么让 B=0，A 随机，而不是两个都 0，或者 A=0？**
这是一个极其精妙的**反向传播数学题**！
假设旁路输出是 $h = x \cdot A \cdot B$。我们要用链式法则算梯度（让参数更新）：
* A 的梯度： $\frac{\partial Loss}{\partial A} \propto B$
* B 的梯度： $\frac{\partial Loss}{\partial B} \propto A$

* **如果 A 和 B 全是 0：** 那么 A 和 B 的梯度在第一步全都是 0！优化器直接死锁，模型永远学不到任何东西（对称性被打破失败）。
* **因为必须有一个是非 0 的（随机高斯分布）：** 作者选择让 A 随机，B 全 0。
* **第一步发生了什么？** 因为 B=0，所以 A 的初始梯度是 0（A 在第一步不动）。但是！因为 A 是随机数（非 0），所以 **B 在第一步获得了非 0 的梯度，B 被更新了！**
* **第二步发生了什么？** 到了第二步，B 已经不再是 0 了，于是 A 也开始获得梯度。齿轮完美转动起来！
*(注：其实让 A=0，B 随机在数学上也是完全等价的，作者只是习惯性地把最后一个矩阵设为 0 罢了。)*

---

### 3. 关于缩放因子（Scaling Factor）：它到底在干嘛？

你连问了：缩放因子作用是什么？为什么需要？为什么能起到作用？

**作用是什么？** 代码里 `self.scaling = lora_alpha / r`。当旁路算出结果后，会乘以这个系数。

**为什么需要它？**
假设你现在用秩 `r=8` 训练了一个很棒的模型，学习率是 `1e-4`。明天老板让你试试 `r=16`，看能不能更聪明。
如果你直接把秩翻倍，矩阵相乘时，求和的项数就翻倍了，**旁路输出的数值方差（大小）也会跟着变大**！
数值变大意味着什么？意味着你原来的学习率 `1e-4` 可能会导致梯度爆炸，你必须**重新去辛苦地调整学习率**！这太反人类了。

**为什么它能起作用？**
除以 `r`，本质上就是一个**归一化（Normalization）操作**。
当 `r` 翻倍变大时，矩阵算出来的值变大了，但你同时除以了更大的 `r`，就把整体数值的量级“压”回了原来的水平。
这样一来，无论你是用 `r=4` 还是 `r=64`，你的**学习率都可以保持不变**！这极大地解放了炼丹师的双手。（`lora_alpha` 通常固定为一个常数，比如 16 或 32，它更像是一个全局的强度放大器）。

---

### 4. 关于 `lora_dropout`：为什么要有它？
**防止大模型在微调数据上“死记硬背”（过拟合）。**
你拿来微调的数据（比如你公司的私有客服语料），相比于预训练的数据来说，简直是沧海一粟（可能只有几千条）。
大模型极其聪明，面对几千条数据，如果不加限制，它瞬间就能把答案“背”下来，导致在训练集上正确率 100%，一测新问题就翻车。
`Dropout` 就是在训练时，每次随机把输入特征的 5%（`lora_dropout=0.05`）变成 0。逼着 LoRA 的 A 和 B 矩阵去学习更加全局、更加鲁棒的特征，而不是死盯住某几个特定词汇。

---

### 5. 终极一问：数据流向 VS 矩阵相加

你说：“我了解到的是训练两个小矩阵，相加到原参数矩阵上。这和你的数据流向有什么关系？”

**这是数学表象与工程现实的最美结合！**

**数学理论上：**
目标是 $W_{new} = W_0 + B A$
输入 $x$ 乘进去，就是： $y = x \cdot (W_0 + B A)$

**利用分配律拆开：**
$y = x \cdot W_0 + x \cdot (B A)$

看到了吗？
* **方案 1（你的理解，直接改权重）：** 先算 $W_{new} = W_0 + BA$，再算 $y = x \cdot W_{new}$。
* **方案 2（我的代码，数据分流）：** 一路走 $x \cdot W_0$（也就是 `self.pretrained(x)`），另一路走 $x \cdot A \cdot B$（也就是 `lora_B(lora_A(x))`），最后把**结果相加**！

**为什么训练时必须用方案 2（数据流向）？**
因为在当今的大模型微调中，$W_0$ 通常有几百个 GB！为了能在一张显卡上跑起来，我们会把 $W_0$ **量化（压缩）成极其粗糙的 INT4 格式（整数）**！
而你的 LoRA 矩阵 A 和 B 是为了高精度学习，必须保持 **FP16 格式（浮点数）**。
你告诉我，一个高精度的 FP16 矩阵，怎么加到一个粗糙的 INT4 矩阵里去？加进去精度就彻底丢失了！
**所以在训练时，我们绝不能去动原本的 $W_0$ 权重。我们让数据同时流过两个矩阵，在最高精度的激活值（Activations）层面去相加！**

**推理（Inference）时呢？**
当我们训练完了，要上线给用户用了。为了追求极致的速度，我们不想每次算两遍。这时候，我们会把原本 INT4 的 $W_0$ 解压回 FP16，然后真正执行：
`W_0 = W_0 + B * A * scaling`
把小矩阵彻底融合进大矩阵里！这就是所谓的 `merge_weights` 操作。融合完之后，扔掉 A 和 B，只用一个 $W_0$ 去推理。模型不仅学到了新知识，而且推理速度和没加 LoRA 时**完全一模一样，0 延迟增加**！